# OCR Model Comparison — Results Viewer

Visualize transcription quality across four vision-language OCR models on
the same column strips produced by the segmentation pipeline.

**To generate results**, run the comparison script first:
```bash
uv run python -m newspapers.ocr.run_comparison
```

| Model | Size | Provider |
|---|---|---|
| Gemini 2.5 Flash | — | Google API |
| DeepSeek-OCR-2 | 3 B | HF Inference Endpoint |
| LightOnOCR-2 | 1 B | HF Inference Endpoint |
| GLM-OCR | 0.9 B | HF Inference Endpoint |

In [ ]:
from __future__ import annotations
import json, io, base64, sys, html as html_mod
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, HTML

# ── repo root ──────────────────────────────────────────────────────────────
REPO = Path().resolve().parent
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

PROCESSED = REPO / "data" / "processed"
INTERIM = REPO / "data" / "interim"
STRIPS_DIR = INTERIM / "strips"

# Load all cached OCR results
result_files = sorted(INTERIM.glob("ocr_comparison_*.json"))
all_results: dict[str, dict[str, dict[str, str]]] = {}
for f in result_files:
    stem = f.stem.replace("ocr_comparison_", "")
    all_results[stem] = json.loads(f.read_text(encoding="utf-8"))

print(f"Loaded results for {len(all_results)} page(s):")
for stem, data in all_results.items():
    models = set()
    for strip_data in data.values():
        models.update(strip_data.keys())
    print(f"  {stem}: {len(data)} strips, models: {sorted(models)}")

## 1 · Load strip images

Run `analyse_page_structure()` to get strip image paths for display (fast, no API calls).

In [ ]:
from newspapers.segmentation.structure import analyse_page_structure

# Build mapping: page_stem -> {strip_id: image_path}
strip_image_paths: dict[str, dict[str, Path]] = {}

for stem in all_results:
    jpg_path = PROCESSED / f"{stem}.jpg"
    png_path = PROCESSED / f"{stem}.png"
    img_path = png_path if png_path.exists() else jpg_path

    if not img_path.exists():
        print(f"  {stem}: image not found, skipping strip paths")
        continue

    _bounds, strips, _profile, skew = analyse_page_structure(img_path, n_columns_hint=8)
    strip_image_paths[stem] = {s.strip_id: s.image_path for s in strips}
    print(f"  {stem}: {len(strips)} strips, skew={skew:.2f}°")

## 2 · Side-by-side comparison

For each strip: the strip image thumbnail on top, then model transcriptions in columns below.

In [ ]:
def _esc(s: str) -> str:
    return html_mod.escape(s)

def _img_to_b64_thumb(img_path: Path, max_height: int = 300) -> str:
    """Create a base64 thumbnail for inline HTML display."""
    img = Image.open(img_path)
    ratio = max_height / img.height
    thumb = img.resize((int(img.width * ratio), max_height), Image.LANCZOS)
    buf = io.BytesIO()
    thumb.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def render_strip_comparison(page_stem: str, strip_id: str,
                            strip_image_path: Path | None,
                            transcriptions: dict[str, str],
                            max_chars: int = 2000) -> str:
    """HTML block: strip image thumbnail + model transcriptions side by side."""
    model_names = list(transcriptions.keys())
    n = len(model_names)
    col_width = max(20, 90 // n)

    # Strip image thumbnail
    img_html = ""
    if strip_image_path and strip_image_path.exists():
        b64 = _img_to_b64_thumb(strip_image_path)
        img_html = (
            f'<div style="padding:8px; text-align:center; background:#fafafa;">'
            f'<img src="data:image/png;base64,{b64}" style="max-height:300px; border:1px solid #ddd;">'
            f'</div>'
        )

    # Header row with model names
    header = "".join(
        f"<th style='text-align:left; padding:8px; border:1px solid #ccc; "
        f"background:#f5f5f5; font-size:13px; width:{col_width}%;'>"
        f"{_esc(m)}</th>"
        for m in model_names
    )

    # Content row
    cells = []
    for m in model_names:
        txt = transcriptions[m]
        is_err = txt.startswith("[ERROR]")
        display_txt = txt[:max_chars] + ("..." if len(txt) > max_chars else "")
        color = "#c00" if is_err else "#222"
        n_chars = len(txt) if not is_err else 0
        cells.append(
            f"<td style='vertical-align:top; padding:8px; border:1px solid #ccc; "
            f"white-space:pre-wrap; font-family:monospace; font-size:11px; "
            f"color:{color}; line-height:1.4;'>"
            f"<em style=\"color:#888; font-size:10px;\">{n_chars} chars</em><br>"
            f"{_esc(display_txt)}</td>"
        )

    return f"""
    <div style="margin:24px 0; border:2px solid #999; border-radius:8px; overflow:hidden;">
      <div style="background:#e8e8e8; padding:8px 12px; font-size:14px; font-weight:bold;">
        {_esc(page_stem)} &mdash; {_esc(strip_id)}
      </div>
      {img_html}
      <table style="border-collapse:collapse; width:100%; table-layout:fixed;">
        <tr>{header}</tr>
        <tr>{"".join(cells)}</tr>
      </table>
    </div>
    """

# Render all comparisons
html_parts = ["<h2>Side-by-Side Transcription Comparison</h2>"]

for page_stem in sorted(all_results.keys()):
    page_data = all_results[page_stem]
    for sid in sorted(page_data.keys()):
        img_path = strip_image_paths.get(page_stem, {}).get(sid)
        html_parts.append(render_strip_comparison(
            page_stem, sid, img_path, page_data[sid]
        ))

display(HTML("".join(html_parts)))

## 3 · Summary statistics

In [ ]:
rows = []
for page_stem, page_data in all_results.items():
    for sid, transcriptions in page_data.items():
        for model_name, text in transcriptions.items():
            is_error = text.startswith("[ERROR]")
            rows.append({
                "page": page_stem.split("_")[-1],  # short page id
                "strip": sid,
                "model": model_name,
                "chars": len(text) if not is_error else 0,
                "words": len(text.split()) if not is_error else 0,
                "error": is_error,
            })

df = pd.DataFrame(rows)

print("=" * 60)
print("  Aggregate stats per model (across all pages)")
print("=" * 60)
agg = df.groupby("model").agg(
    total_chars=("chars", "sum"),
    total_words=("words", "sum"),
    avg_chars_per_strip=("chars", "mean"),
    strips_processed=("chars", "count"),
    errors=("error", "sum"),
).sort_values("total_chars", ascending=False)
display(agg)

print("\n" + "=" * 60)
print("  Per-strip character counts (pivot)")
print("=" * 60)
df["label"] = df["page"] + " / " + df["strip"]
pivot = df.pivot_table(index="label", columns="model", values="chars", aggfunc="first")
display(pivot)

In [ ]:
# Bar chart: characters per model per strip (across all pages)
if not df.empty and not df["chars"].eq(0).all():
    fig, ax = plt.subplots(figsize=(14, 6))
    pivot.plot(kind="bar", ax=ax, edgecolor="white", linewidth=0.5)
    ax.set_ylabel("Characters")
    ax.set_xlabel("")
    ax.set_title("Transcription length by model and strip (all pages)")
    ax.legend(title="Model", fontsize=8, title_fontsize=9)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    plt.show()